# ProtCell protein-panel search

Use frozen ProtCell reconstruction to choose a compact protein panel. Test information retained by that panel through profile completion and measured-only AML classification. The pretrained model remains frozen during search; no model training is required here.

Run the cells in order with the `protein_llm` kernel. Edit the data and checkpoint paths in the setup cell. The `RUN_*` switches enable the corresponding fine-tuning, prediction or analysis steps.

In [ ]:
from pathlib import Path
import os
import sys
import tempfile

PROJECT_ROOT = Path(os.environ.get("PROTCELL_PROJECT_ROOT", str(Path.cwd()))).expanduser().resolve()
if not (PROJECT_ROOT / "train.py").is_file() and (PROJECT_ROOT.parent / "train.py").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "train.py").is_file(), "Set PROTCELL_PROJECT_ROOT to your ProtCell clone before starting JupyterLab."
OUTPUT_ROOT = Path(os.environ.get("PROTCELL_OUTPUT_DIR", str(PROJECT_ROOT / "results/outputs")))
OUTPUT_DIR = OUTPUT_ROOT / "panel_search"
FIGURE_DIR = OUTPUT_DIR / "figures"
RESULT_DIR = OUTPUT_DIR / "results"
MODEL_DIR = OUTPUT_DIR / "models"
for directory in [OUTPUT_DIR, FIGURE_DIR, RESULT_DIR, MODEL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT_ROOT))
sys.dont_write_bytecode = True
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
os.chdir(OUTPUT_DIR) # trainer logs and new files stay with this experiment

old_cache_settings = {key: os.environ.get(key) for key in ["MPLCONFIGDIR", "NUMBA_CACHE_DIR", "XDG_CACHE_HOME"]}
plot_cache = tempfile.TemporaryDirectory(prefix="protcell-panel_search-", dir=OUTPUT_DIR)
os.environ["MPLCONFIGDIR"] = plot_cache.name
os.environ["NUMBA_CACHE_DIR"] = str(Path(plot_cache.name) / "numba")
os.environ["XDG_CACHE_HOME"] = str(Path(plot_cache.name) / "xdg")

import json
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import sparse, stats
from tqdm.auto import tqdm
from IPython.display import display
from model.model import ProteinLanguageModel
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score
plt.rcParams.update({"pdf.fonttype": 42, "ps.fonttype": 42, "font.family": "sans-serif"})
sc.settings.figdir = FIGURE_DIR
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DATA_ROOT = Path(os.environ.get("PROTCELL_DATA_ROOT", str(PROJECT_ROOT / "data"))).expanduser().resolve()
BASE_MODEL_PATH = PROJECT_ROOT / "checkpoints/basemodel/large-scale-new-10/model_369999.pt"
print("Device:", DEVICE)
print("Outputs:", OUTPUT_DIR)

## Read discovery, selection and test data

Sparks2023 (Dataset101) uses the frozen, donor-disjoint discovery, selection and test splits of the paper. Inputs are already CLR processed. Enable `RUN_PROTCELL_SEARCH` to repeat the 200-run search and `USE_NEW_PANEL` to evaluate its winner.


In [ ]:
INPUT_DIR = DATA_ROOT / "panel_search"
PANEL_DEFINITION_PATH = PROJECT_ROOT / "results/panel_definition.json"
discovery = ad.read_h5ad(INPUT_DIR / "dataset101_discovery.h5ad")
selection = ad.read_h5ad(INPUT_DIR / "dataset101_selection.h5ad")
test = ad.read_h5ad(INPUT_DIR / "dataset101_test.h5ad")
proteins = discovery.var_names.tolist()
assert selection.var_names.tolist() == proteins == test.var_names.tolist()
donor_col, visit_col = "alt.subject.id", "visit"
donor_sets = [set(data.obs[donor_col].astype(str)) for data in [discovery, selection, test]]
assert all(donor_sets[i].isdisjoint(donor_sets[j]) for i, j in [(0, 1), (0, 2), (1, 2)])
discovery_matrix = np.asarray(discovery.X.toarray() if sparse.issparse(discovery.X) else discovery.X, dtype=np.float32)
selection_matrix = np.asarray(selection.X.toarray() if sparse.issparse(selection.X) else selection.X, dtype=np.float32)
test_matrix = np.asarray(test.X.toarray() if sparse.issparse(test.X) else test.X, dtype=np.float32)
statistics = np.load(INPUT_DIR / "discovery_protein_statistics.npz")
center, scale = statistics["means"].astype(np.float32), statistics["scales"].astype(np.float32)
scale = np.where(scale > 1e-8, scale, 1.0)
statistics.close()
ANCHORS = ["CD3", "CD4", "CD19", "CD14", "KLRD1", "FCGR3A,FCGR3B"]
BUDGET = 60
RUN_PROTCELL_SEARCH = True # optional: repeat the paper's 200-run search
RUN_COMPLETION = True
RUN_AML = True
USE_NEW_PANEL = True # False reproduces the frozen panel used in Fig. 2j
display(pd.DataFrame({"split": ["discovery", "selection", "test"],
                      "cells": [len(discovery), len(selection), len(test)],
                      "donors": [len(values) for values in donor_sets]}))

## Load the frozen ProtCell model

The pretrained model supplies the search objective; its parameters are not updated during panel selection.

In [ ]:
model = ProteinLanguageModel.load_model_from_checkpoint(str(BASE_MODEL_PATH), device=DEVICE).to(DEVICE)
model.eval()
for parameter in model.parameters():
    parameter.requires_grad_(False)
protein_ids = torch.tensor(model.tokenizer.encode(proteins), dtype=torch.long, device=DEVICE)
display(model)

## The masked reconstruction objective
Compute standardized reconstruction MSE for omitted proteins.

In [ ]:
def panel_mse(panel, matrix, batch_size=128):
    omitted = np.array([name not in set(panel) for name in proteins])
    total_squared_error = 0.0
    with torch.no_grad():
        for start in range(0, len(matrix), batch_size):
            batch = torch.as_tensor(matrix[start:start + batch_size], dtype=torch.float32, device=DEVICE)
            ids = protein_ids.expand(len(batch), -1)
            mask = torch.as_tensor(omitted, dtype=torch.bool, device=DEVICE).expand(len(batch), -1)
            predicted, _ = model(input_ids=ids, expr_values=batch, mask=mask)
            predicted = predicted.reshape(len(batch), int(omitted.sum())).cpu().numpy()
            total_squared_error += np.sum(((np.asarray(matrix[start:start + len(batch), omitted], dtype=float)
                                           - np.asarray(predicted, dtype=float)) / scale[omitted]) ** 2)
    return float(total_squared_error / (len(matrix) * omitted.sum()))

## Search ProtCell panels (optional)

Run 20 cell resamples × 10 search seeds, 40 initial panels, then two rounds of 24 single-marker exchanges. Each run winner is scored on the remaining selection cells; the lowest-error winner becomes the final panel. Test data are not scored here. Fig. 2j evaluates this single frozen panel, rather than averaging test performance across 200 search runs.


In [ ]:
if RUN_PROTCELL_SEARCH:
    from results.panel_search_utils import stratified_search_split, generate_random_panels
    search_winners = []
    for cell_index, cell_seed in enumerate(range(11001, 11021), 1):
        split = stratified_search_split(selection.obs, donor_col, visit_col, 32, cell_seed)
        search_matrix = selection_matrix[split.search_indices]
        heldout_matrix = selection_matrix[split.heldout_indices]
        for search_index, search_seed in enumerate(range(21001, 21011), 1):
            run_id = f"cell_{cell_index:02d}_search_{search_index:02d}"
            candidates = generate_random_panels(proteins, BUDGET, ANCHORS, 40, search_seed)
            history, evaluated = [], set()
            for panel in candidates:
                key = tuple(panel)
                if key in evaluated:
                    continue
                evaluated.add(key)
                history.append(dict(stage="random", round=0, score=panel_mse(panel, search_matrix), panel="|".join(panel)))
            rng = np.random.default_rng(search_seed + 1)
            for round_number in [1, 2]:
                current = min(history, key=lambda row: row["score"])["panel"].split("|")
                removable = [protein for protein in current if protein not in ANCHORS]
                available = [protein for protein in proteins if protein not in current]
                accepted = attempts = 0
                while accepted < 24 and attempts < 24 * 20:
                    attempts += 1
                    removed, added = str(rng.choice(removable)), str(rng.choice(available))
                    chosen = (set(current) - {removed}) | {added}
                    proposal = tuple(protein for protein in proteins if protein in chosen)
                    if proposal in evaluated:
                        continue
                    evaluated.add(proposal); accepted += 1
                    history.append(dict(stage="swap", round=round_number,
                                        score=panel_mse(proposal, search_matrix), panel="|".join(proposal)))
            history = pd.DataFrame(history).sort_values(["score", "stage", "round"]).reset_index(drop=True)
            winner = history.iloc[0].panel.split("|")
            history.to_csv(RESULT_DIR / f"{run_id}_search_history.csv", index=False)
            search_winners.append(dict(run_id=run_id, panel="|".join(winner),
                                       selection_mse=panel_mse(winner, heldout_matrix)))
    search_winners = pd.DataFrame(search_winners)
    search_winners.to_csv(RESULT_DIR / "new_protcell_search_winners.csv", index=False)
    new_protcell_panel = search_winners.loc[search_winners.selection_mse.idxmin(), "panel"].split("|")
    display(search_winners.sort_values("selection_mse").head())

## Select the ProtCell panel and the Fig. 2j evaluation targets

For reproduction, load the published 60-protein ProtCell panel and the exact evaluation target lists: 39 proteins in Sparks2023 and 80 in Beneyto-Calabuig2023. 

`USE_NEW_PANEL=True` instead evaluates the new search winner. If it differs from the published panel, its results are reported as a new experiment. It must still leave every frozen target unmeasured.


In [ ]:
panel_definition = json.loads(PANEL_DEFINITION_PATH.read_text())
published_panel = panel_definition["panel"]
if USE_NEW_PANEL:
    assert "new_protcell_panel" in globals(), "Run the search cell before setting USE_NEW_PANEL=True."
    selected_panel = new_protcell_panel
else:
    selected_panel = published_panel
assert len(selected_panel) == len(set(selected_panel)) == BUDGET
assert set(ANCHORS).issubset(selected_panel) and set(selected_panel).issubset(proteins)
panels = {"ProtCell": selected_panel}
is_published_panel = selected_panel == published_panel
completion_targets = panel_definition["targets"]
assert len(completion_targets["Dataset101"]) == 39
assert len(completion_targets["Dataset107"]) == 80
assert len(test) == 24000, "Fig. 2j requires 24,000 Sparks2023 test cells."
for dataset, targets in completion_targets.items():
    assert len(targets) == len(set(targets))
    assert set(targets).isdisjoint(selected_panel), f"{dataset}: the panel measures a frozen evaluation target."
(RESULT_DIR / "ProtCell_selected_panel.json").write_text(json.dumps(selected_panel, indent=2))
print("Evaluating published panel:", is_published_panel)
display(pd.Series(selected_panel, name="Selected protein"))


## Prepare transferred data for panel completion

Beneyto-Calabuig2023 (Dataset107) uses Cohort A for decoder fitting and all 45,903 Cohort B cells for evaluation.


In [ ]:
transfer = ad.read_h5ad(DATA_ROOT / "CLR_data/Dataset107.h5ad")
transfer_proteins = transfer.var_names.tolist()
transfer_matrix = np.asarray(transfer.X.toarray() if sparse.issparse(transfer.X) else transfer.X, dtype=np.float32)
cohort_a = transfer.obs.cohort.astype(str).eq("Cohort A").to_numpy()
cohort_b = transfer.obs.cohort.astype(str).eq("Cohort B").to_numpy()
assert set(transfer.obs.loc[cohort_a, "patient"].astype(str)).isdisjoint(transfer.obs.loc[cohort_b, "patient"].astype(str))
assert int(cohort_a.sum()) == 41744 and int(cohort_b.sum()) == 45903
if is_published_panel:
    assert [p for p in selected_panel if p in transfer_proteins] == panel_definition["mapped_panel"]
completion_inputs = {
    "Dataset101": (discovery_matrix, test_matrix, proteins),
    "Dataset107": (transfer_matrix[cohort_a], transfer_matrix[cohort_b], transfer_proteins),
}
display(pd.DataFrame({name: [len(set(panel).intersection(proteins)), len(set(panel).intersection(transfer_proteins))]
                      for name, panel in panels.items()}, index=["Dataset101", "Dataset107"]))
completion_protocol = {
    "target_scope": "fig2j_common_targets",
    "is_published_panel": is_published_panel,
    "selected_panel": selected_panel,
    "target_lists": completion_targets,
    "target_sources": {"Dataset101": str(PANEL_DEFINITION_PATH), "Dataset107": str(PANEL_DEFINITION_PATH)},
    "seed": SEED,
    "datasets": {},
}
for dataset, (train_matrix, evaluation_matrix, names) in completion_inputs.items():
    assert set(completion_targets[dataset]).issubset(names)
    completion_protocol["datasets"][dataset] = {
        "train_cells": len(train_matrix), "test_cells": len(evaluation_matrix),
        "measured_proteins": [p for p in selected_panel if p in names],
        "n_targets": len(completion_targets[dataset]),
    }
(RESULT_DIR / "new_completion_protocol.json").write_text(json.dumps(completion_protocol, indent=2))


## Test information retained by the ProtCell panel

Fit ridge, nearest-neighbour and extremely randomized tree decoders using the parameters, input feature order and random seed specified below. Each decoder is trained to reconstruct all proteins omitted by ProtCell; only the fixed Fig. 2j targets contribute to evaluation. Keeping the full training output set also preserves the multi-output tree objective.

Compute Spearman correlation and RMSE across test cells for each target protein. Fig. 2j reports the mean of the protein-wise correlations and the median of protein-wise RMSE values. These are cross-protein summaries, not averages over search seeds.


In [ ]:
new_completion_summary = None
if RUN_COMPLETION:
    completion_rows, completion_summaries = [], []
    for dataset, (train_matrix, evaluation_matrix, names) in completion_inputs.items():
        omitted_targets = completion_targets[dataset]
        assert len(omitted_targets) > 0, "The selected panel leaves no omitted targets."
        evaluation_panels = dict(panels)
        for method, panel in evaluation_panels.items():
            measured_indices = [names.index(protein) for protein in panel if protein in names]
            target_indices = [i for i, name in enumerate(names) if name not in panel]
            target_names = [names[i] for i in target_indices]
            omitted_columns = [target_names.index(protein) for protein in omitted_targets]
            target_truth = evaluation_matrix[:, [names.index(protein) for protein in omitted_targets]]
            for decoder in ["ridge", "knn", "extra_trees"]:
                print(f"{dataset}: fitting {decoder}; {len(measured_indices)} inputs, {len(omitted_targets)} evaluation targets", flush=True)
                dtype = np.float32 if decoder == "extra_trees" else np.float64
                x_train = train_matrix[:, measured_indices].astype(dtype)
                y_train = train_matrix[:, target_indices].astype(dtype)
                x_test = evaluation_matrix[:, measured_indices].astype(dtype)
                if decoder != "ridge":
                    max_cells = (5000 if dataset == "Dataset101" else 8000) if decoder == "extra_trees" else 5000
                    if len(x_train) > max_cells:
                        selected = np.sort(np.random.default_rng(SEED).choice(len(x_train), max_cells, replace=False))
                        x_train, y_train = x_train[selected], y_train[selected]
                if decoder in ["ridge", "knn"]:
                    x_mean, x_scale = x_train.mean(axis=0), x_train.std(axis=0)
                    x_scale = np.where(x_scale > 1e-8, x_scale, 1.0)
                    x_train, x_test = (x_train - x_mean) / x_scale, (x_test - x_mean) / x_scale
                if decoder == "ridge":
                    y_mean, y_scale = y_train.mean(axis=0), y_train.std(axis=0)
                    y_scale = np.where(y_scale > 1e-8, y_scale, 1.0)
                    decoder_model = Ridge(alpha=10.0)
                    decoder_model.fit(x_train, (y_train - y_mean) / y_scale)
                    prediction = decoder_model.predict(x_test) * y_scale + y_mean
                elif decoder == "knn":
                    decoder_model = KNeighborsRegressor(n_neighbors=min(20, len(x_train)), weights="distance", n_jobs=-1)
                    decoder_model.fit(x_train, y_train)
                    prediction = decoder_model.predict(x_test)
                else:
                    decoder_model = ExtraTreesRegressor(n_estimators=150 if dataset == "Dataset101" else 300,
                                                        min_samples_leaf=2, max_features=1.0, n_jobs=-1, random_state=SEED)
                    decoder_model.fit(x_train, y_train)
                    prediction = decoder_model.predict(x_test)
                prediction = np.asarray(prediction, dtype=np.float32)[:, omitted_columns]
                metric_dtype = np.float32 if dataset == "Dataset101" else np.float64
                metric_truth = np.asarray(target_truth, dtype=metric_dtype)
                metric_prediction = np.asarray(prediction, dtype=metric_dtype)
                protein_rmse = np.sqrt(np.mean((metric_prediction - metric_truth) ** 2, axis=0))
                target_rows = []
                for j, protein in enumerate(omitted_targets):
                    row = dict(dataset=dataset, method=method, decoder=decoder, protein=protein,
                               spearman=stats.spearmanr(target_truth[:, j], prediction[:, j]).statistic,
                               rmse=float(protein_rmse[j]), target_scope="fig2j_common_targets")
                    target_rows.append(row); completion_rows.append(row)
                target_table = pd.DataFrame(target_rows)
                completion_summaries.append(dict(dataset=dataset, method=method, decoder=decoder,
                    mean_protein_spearman=target_table.spearman.mean(), median_protein_rmse=target_table.rmse.median(),
                    measured_markers=len(measured_indices), targets=len(omitted_targets),
                    test_cells=len(evaluation_matrix), target_scope="fig2j_common_targets"))
    pd.DataFrame(completion_rows).to_csv(RESULT_DIR / "new_completion_protein_metrics.csv", index=False)
    new_completion_summary = pd.DataFrame(completion_summaries)
    new_completion_summary.to_csv(RESULT_DIR / "new_completion_summary.csv", index=False)
    display(new_completion_summary)

## Plot completion from the ProtCell panel

Display and plot the completion summary calculated above. Each decoder predicts the fixed evaluation targets from the same selected measurements.


In [ ]:
if RUN_COMPLETION:
    assert new_completion_summary is not None, "Run the completion evaluation cell before plotting."
    completion_summary = new_completion_summary.copy()
    completion_summary = completion_summary.loc[completion_summary.method.eq("ProtCell")].copy()
    if len(completion_summary):
        display(completion_summary)
        fig, axes = plt.subplots(1, 2, figsize=(10, 4))
        for ax, metric in zip(axes, ["mean_protein_spearman", "median_protein_rmse"]):
            sns.barplot(data=completion_summary, x="decoder", y=metric, hue="dataset", errorbar=None, ax=ax)
            ax.set_title(f"ProtCell panel: {metric}")
        fig.tight_layout(); fig.savefig(FIGURE_DIR / "ProtCell_panel_completion.pdf", bbox_inches="tight"); plt.show()
else:
    print("Enable RUN_COMPLETION to compute and plot panel-completion metrics.")

## Prepare measured-only AML classification data

Sample up to 1,000 labelled myeloid cells per patient and status. Train on Cohort A; retain Cohort B exclusively for patient-level testing. Sampling is independent of panel selection.

In [ ]:
if RUN_AML:
    myeloid_types = ["Immature", "Early myeloid", "Monocytes", "Erythroid", "Dendritic"]
    eligible = transfer.obs.status.astype(str).isin(["healthy", "leukemic"]) & transfer.obs.celltype.astype(str).isin(myeloid_types)
    
    metadata = transfer.obs.loc[eligible, ["patient", "status"]].copy()
    metadata["position"] = np.flatnonzero(eligible.to_numpy())
    positions = []
    for offset, (_, group) in enumerate(metadata.groupby(["patient", "status"], observed=True, sort=True)):
        group_positions = group.position.to_numpy()
        positions.extend(np.random.default_rng(SEED + offset).choice(group_positions, min(1000, len(group_positions)), replace=False))
    sampled_positions = np.sort(np.array(positions, dtype=int))
    train_indices = sampled_positions[cohort_a[sampled_positions]]
    test_indices = sampled_positions[cohort_b[sampled_positions]]
    train_y = transfer.obs.iloc[train_indices].status.astype(str).eq("leukemic").to_numpy().astype(int)
    test_y = transfer.obs.iloc[test_indices].status.astype(str).eq("leukemic").to_numpy().astype(int)
    train_patients = transfer.obs.iloc[train_indices].patient.astype(str).to_numpy()
    test_metadata = transfer.obs.iloc[test_indices].copy()
    display(test_metadata.groupby("patient", observed=True).size())

## Select the downstream classifier penalty

Choose C using stratified patient-grouped cross-validation on the training cohort. Held-out patient labels are used only to evaluate predictions.

In [ ]:
if RUN_AML:
    full_train_x = transfer_matrix[train_indices].astype(float)
    n_splits = min(5, len(np.unique(train_patients)), *(len(np.unique(train_patients[train_y == label])) for label in [0, 1]))
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    c_scores = []
    for c_value in [0.01, 0.1, 1.0, 10.0]:
        out_of_fold = np.full(len(train_y), np.nan)
        for fit_indices, validation_indices in splitter.split(full_train_x, train_y, train_patients):
            if len(np.unique(train_y[fit_indices])) < 2 or len(np.unique(train_y[validation_indices])) < 2:
                continue
            classifier = make_pipeline(StandardScaler(), LogisticRegression(
                C=c_value, penalty="l2", class_weight="balanced", solver="liblinear", max_iter=5000, random_state=SEED))
            classifier.fit(full_train_x[fit_indices], train_y[fit_indices])
            out_of_fold[validation_indices] = classifier.predict_proba(full_train_x[validation_indices])[:, 1]
        valid = np.isfinite(out_of_fold)
        c_scores.append(dict(C=c_value, auroc=roc_auc_score(train_y[valid], out_of_fold[valid])))
    c_scores = pd.DataFrame(c_scores)
    selected_c = float(c_scores.loc[c_scores.auroc.idxmax(), "C"])
    c_scores.to_csv(RESULT_DIR / "new_AML_training_CV.csv", index=False)
    display(c_scores)

## Map strict 40-marker panels and fit logistic regression

The inputs contain measured values only.

In [ ]:
aml_patient_metrics = None
if RUN_AML:
    aml_panels = {}
    for name, source_panel in panels.items():
        mapped = [protein for protein in source_panel if protein in transfer_proteins][:40]
        if len(mapped) < 40:
            mapped += [protein for protein in proteins if protein in transfer_proteins and protein not in mapped][:40 - len(mapped)]
        assert len(mapped) == len(set(mapped)) == 40
        aml_panels[name] = mapped
    aml_rows = []
    for method, panel in aml_panels.items():
        columns = [transfer_proteins.index(protein) for protein in panel]
        dtype = np.float32
        x_train = transfer_matrix[np.ix_(train_indices, columns)].astype(dtype)
        x_test = transfer_matrix[np.ix_(test_indices, columns)].astype(dtype)
        classifier_seed = SEED
        classifier = make_pipeline(StandardScaler(), LogisticRegression(
            C=selected_c, penalty="l2", class_weight="balanced", solver="liblinear", max_iter=5000, random_state=classifier_seed))
        classifier.fit(x_train, train_y)
        probabilities = classifier.predict_proba(x_test)[:, 1]
        for patient, positions in test_metadata.groupby("patient", observed=True, sort=True).indices.items():
            labels = test_y[positions]
            if len(np.unique(labels)) != 2:
                continue
            aml_rows.append(dict(method=method, patient=str(patient), n_cells=len(positions),
                auroc=roc_auc_score(labels, probabilities[positions]), auprc=average_precision_score(labels, probabilities[positions])))
    aml_patient_metrics = pd.DataFrame(aml_rows)
    aml_summary = aml_patient_metrics.groupby("method", observed=True)[["auroc", "auprc"]].median()
    aml_patient_metrics.to_csv(RESULT_DIR / "new_AML_patient_metrics.csv", index=False)
    aml_summary.to_csv(RESULT_DIR / "new_AML_patient_medians.csv")
    (RESULT_DIR / "new_AML_evaluation_panels.json").write_text(json.dumps(aml_panels, indent=2))
    display(aml_summary)

## Plot AML performance of the ProtCell panel

Summarize the held-out patient-level predictions from the preceding measured-only test.

In [ ]:
if RUN_AML:
    assert aml_patient_metrics is not None, "Run the AML classification cell before plotting."
    patient_metrics = aml_patient_metrics.copy()
    patient_metrics = patient_metrics.loc[patient_metrics.method.eq("ProtCell")].copy()
    if len(patient_metrics):
        display(patient_metrics)
        display(patient_metrics[["auroc", "auprc"]].median().rename("Patient median"))
        fig, ax = plt.subplots(figsize=(6, 4))
        patient_metrics.set_index("patient")[["auroc", "auprc"]].plot.bar(ax=ax, color=["#659AAF", "#DF9A6B"])
        ax.set_ylim(0, 1); ax.set_title("ProtCell-selected 40-protein panel")
        fig.tight_layout(); fig.savefig(FIGURE_DIR / "ProtCell_panel_AML.pdf", bbox_inches="tight"); plt.show()
else:
    print("Enable RUN_AML to compute and plot the panel’s patient-level classification results.")

## Release temporary files

Keep the final tables, figures and trained models; delete this notebook’s temporary plotting/JIT caches.

In [ ]:
plt.close("all")
plot_cache.cleanup()
for key, previous in old_cache_settings.items():
    if previous is None:
        os.environ.pop(key, None)
    else:
        os.environ[key] = previous
print("Temporary caches removed.")